*0.4 Deep learning basics*

# Transformer: multi-head attention

**The situation.** One attention (0.2, item 7) gives each token one way of looking at the sentence. But "it" needs to find its noun, "not" needs to find what it negates, and a verb needs its subject — several different relations at once. One set of weights has to average them and does none well.

**Multi-head attention.** Split the model's vector into *h* pieces (heads), run attention separately on each with its own Q/K/V projections, then concatenate the results and mix them with one final linear layer. Each head can learn a different relation. It costs the same as one big attention; the split is free.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(0)
d_model, heads = 64, 4
attention = nn.MultiheadAttention(
    embed_dim=d_model, num_heads=heads, batch_first=True, bias=False
).eval()
tokens = torch.randn(1, 6, d_model)  # six tokens

with torch.no_grad():
    library_output, weights = attention(tokens, tokens, tokens, average_attn_weights=False)
print(
    "output:",
    tuple(library_output.shape),
    "| attention weights:",
    tuple(weights.shape),
    "→ (batch, heads, query token, key token)",
)

# The same thing by hand from the layer's own weights, to see the split.
with torch.no_grad():
    q_weight, k_weight, v_weight = attention.in_proj_weight.chunk(3)
    q, k, v = tokens @ q_weight.T, tokens @ k_weight.T, tokens @ v_weight.T
    head_size = d_model // heads
    q = q.view(1, 6, heads, head_size).transpose(1, 2)  # (batch, heads, tokens, head_size)
    k = k.view(1, 6, heads, head_size).transpose(1, 2)
    v = v.view(1, 6, heads, head_size).transpose(1, 2)
    per_head = F.scaled_dot_product_attention(q, k, v)  # attention on every head at once
    merged = per_head.transpose(1, 2).reshape(1, 6, d_model)  # concatenate heads
    by_hand = merged @ attention.out_proj.weight.T  # final mix
print("by hand equals the library:", torch.allclose(by_hand, library_output, atol=1e-5))
assert torch.allclose(by_hand, library_output, atol=1e-5)

output: (1, 6, 64) | attention weights: (1, 4, 6, 6) → (batch, heads, query token, key token)
by hand equals the library: True


**Reading the output.** The weights tensor has one full `tokens × tokens` map *per head* — four different ways of looking. Reproducing the layer by hand shows the recipe: project, split into heads, attend, concatenate, mix.

**Each head looks somewhere different.** Where does token 0 look, per head?

In [3]:
for head in range(heads):
    row = weights[0, head, 0]
    print(
        
            f"head {head}: token 0 attends most to token {int(row.argmax())} with weight "
            f"{row.max():.2f}"
        
    )
assert weights[0].sum(dim=-1).allclose(torch.ones(heads, 6))

head 0: token 0 attends most to token 3 with weight 0.37
head 1: token 0 attends most to token 5 with weight 0.27
head 2: token 0 attends most to token 4 with weight 0.37
head 3: token 0 attends most to token 0 with weight 0.23


```
x (64) ─▶ Q,K,V ─▶ split ─▶ head 0 (16)  attention ─┐
                          ─▶ head 1 (16)  attention ─┼─▶ concat (64) ─▶ out_proj ─▶ (64)
                          ─▶ head 2 (16)  attention ─┤
                          ─▶ head 3 (16)  attention ─┘
```

**The rule to remember.** Multi-head = several small attentions in parallel, each free to learn a different relation, merged at the end. `d_model` must divide by the head count.

| Use it when | Don't when | Instead use |
|---|---|---|
| every transformer layer | — | — |

**Watch out**
- `average_attn_weights=False` to see per-head maps; the default averages them and hides the point.
- In served models, K and V per head are what the KV cache stores; grouped-query attention shares them across heads to shrink it.
- More heads is not free accuracy: each head gets a smaller slice; 8–16 heads at 64–128 per head is the usual range.